# 架空データで jpomop を最初から最後まで動かす

架空の患者 100 人のデータを使い、jpomop でファイルの読み込みから OMOP CDM への変換まで試します。
2つの入力ファイルの中身、仕様との照合結果、変換後の8つの表を順に見ます。

このノートブックに出てくる患者・病院・薬のコード・診療行為のコードは、すべて架空です。
ICD-10 のコードだけは実在しますが、実在の患者のものではありません。
動かすために、病院のデータ、マスター、Athena からのダウンロードを用意する必要はありません。

jpomop は開発の初期段階にあります（バージョン 0.0.1）。
病院の実データでは一度も動かしていません。
このノートブックで確かめられるのは、仕様どおりに作ったファイルを jpomop がどう扱うかです。
実データでどこまで正しく動くかは、まだ分かりません。

次のコマンドで動かせます。

```sh
git clone https://github.com/kojikojikojikoji/jpomop
cd jpomop
pip install -e . pandas jupyter
jupyter lab examples/walkthrough_ja.ipynb
```

英語版は `walkthrough_en.ipynb` です。

## 1. 準備

ファイルはすべて一時フォルダに作り、最後のセルでフォルダごと消します。

In [1]:
from pathlib import Path
import tempfile

import duckdb
import pandas as pd

import jpomop

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

SEED = 31                # see section 2
FOLLOW = "0000000048"    # the patient we follow (data_id in the DPC files)
FOLLOW_PERSON = 48       # the same patient in the CDM tables

work = Path(tempfile.mkdtemp(prefix="jpomop_"))
print("jpomop", jpomop.__version__)

jpomop 0.0.1


## 2. 架空データを作る

`generate` は 2026 年 6 月分のファイルを2つ書き出します。
同じ seed を渡せば、毎回同じ内容のファイルができます。

- `yoshiki1.txt` は様式1 です。入院ごとに、患者・入院・退院・病名・手術が入っています。
- `ef.txt` は入院EF統合ファイルです。算定した項目が、1日1項目につき1行で入っています。

どちらもタブ区切りのテキストで、日本語はシフト JIS です。DPC 調査の実施説明資料が定めている形式に合わせています。jpomop は、Windows のシフト JIS である CP932 として読み込みます。

このノートブックでは seed に 31 を使います。
この seed では、1人の患者（データ識別番号 0000000048）の1回の入院に、病棟の移動、手術、持参薬、退院時処方、入院後に発症した病名がそろいます。
`jpomop synth` の既定の seed では、別のファイルができます。

In [2]:
from jpomop.synth import generate

m = generate(work / "synth", n_patients=100, seed=SEED)
print(m.n_patients, "patients,", m.n_stays, "stays")
for p in (m.yoshiki1, m.ef):
    print(p.name, p.stat().st_size, "bytes,", len(p.read_bytes().splitlines()), "lines")

100 patients, 120 stays
yoshiki1.txt 73350 bytes, 900 lines
ef.txt 337757 bytes, 1840 lines


## 3. 様式1 の中身

様式1 のファイルでは、1回の入院が複数の行に分かれます。
1行は 17 項目です。前の8項目は固定で、後ろの9項目（下の表の payload1〜payload9）の意味は行ごとに変わります。
後ろの9項目に何が入っているかは、`code` の列で決まります。

データ識別番号が 0000000048 の患者の行を出します。
以降は、この患者の変換結果を追います。

In [3]:
from jpomop.spec import load_spec

y1_spec = load_spec("dpc_yoshiki1")
lines = m.yoshiki1.read_bytes().decode("cp932").splitlines()
raw_y1 = pd.DataFrame([ln.split("\t") for ln in lines], columns=y1_spec.field_names())
raw_y1[raw_y1["data_id"] == FOLLOW]

,facility_code,data_id,admission_date,stay_no,summary_no,code,version,seq,payload1,payload2,payload3,payload4,payload5,payload6,payload7,payload8,payload9
413,999999999,0000000048,20260605,0,0,A000010,20140401,0,19420906,1,1000001,,,,,,
414,999999999,0000000048,20260605,0,0,A000020,20140401,0,20260605,1,,,100,0,,,
415,999999999,0000000048,20260605,0,0,A000030,20140401,0,20260610,1,4,0,,,,,
416,999999999,0000000048,20260605,0,0,A006010,20140401,0,,C189,,9000006,,,,,結腸癌
417,999999999,0000000048,20260605,0,0,A006020,20140401,0,,C189,,9000006,,,,,結腸癌
418,999999999,0000000048,20260605,0,0,A006030,20140401,0,,C189,,9000006,,,,,結腸癌
419,999999999,0000000048,20260605,0,0,A006040,20140401,1,,E785,,9000103,,,,,脂質異常症
420,999999999,0000000048,20260605,0,0,A006040,20140401,2,,N189,,9000104,,,,,慢性腎臓病
421,999999999,0000000048,20260605,0,0,A007010,20140401,1,20260606,K9999,,,,,,,架空手術
422,999999999,0000000048,20260605,0,0,A006050,20140401,1,,J690,,9000201,,,,,誤嚥性肺炎


各行は次のように読みます。

| `code` | 行の内容 | 上の表で使っている後ろの9項目 |
|---|---|---|
| A000010 | 患者 | 1: 生年月日、2: 性別、3: 郵便番号 |
| A000020 | 入院 | 1: 入院年月日、2: 入院経路 |
| A000030 | 退院 | 1: 退院年月日、2: 退院先、3: 退院時転帰、4: 24時間以内の死亡 |
| A006010 | 主傷病 | 2: ICD-10、4: 傷病名コード、5: 修飾語コード、9: 傷病名 |
| A006020 | 入院の契機となった傷病 | A006010 と同じ |
| A006030 | 医療資源を最も投入した傷病 | A006010 と同じ |
| A006040 | 入院時併存症 | A006010 と同じ |
| A006050 | 入院後発症疾患 | A006010 と同じ |
| A007010 | 手術 | 1: 手術日、2: 点数表コード、9: 手術名 |

この患者は 2026 年 6 月 5 日から 10 日まで入院しています。
主傷病は C189（結腸癌）で、同じコードが3行に入っています。
入院時から E785 と N189 があり、入院後に J690（誤嚥性肺炎）を発症しています。
手術は 6 月 6 日に1回受けています。

修飾語コード（5 番目の項目）は、この患者では空です。
ここに 8002 が入ると「の疑い」を表します。例は 8 章で見ます。

`summary_no` は、実施説明資料の「統括診療情報番号」です。
0 は、入院全体を1枚にまとめた様式1 を表します。
0 以外の例は 10 章で見ます。

jpomop では、ファイルの形式を YAML ファイルで定義しています。
項目ごとに、位置・型・桁数・実施説明資料のページ番号を書いてあります。

In [4]:
pd.DataFrame(
    [(f.col, f.name, f.name_ja, f.type, f.length, f.page) for f in y1_spec.fields],
    columns=["col", "name", "name_ja", "type", "length", "manual_page"],
).astype({"length": "Int64"})

,col,name,name_ja,type,length,manual_page
0,1,facility_code,施設コード,digits,9,22
1,2,data_id,データ識別番号,digits,10,22
2,3,admission_date,入院年月日,date8,8,22
3,4,stay_no,回数管理番号,string,<NA>,22
4,5,summary_no,統括診療情報番号,string,<NA>,22
5,6,code,コード,string,7,22
6,7,version,バージョン,date8,8,22
7,8,seq,連番,integer,<NA>,22
8,9,payload1,ペイロード1,string,<NA>,22
9,10,payload2,ペイロード2,string,<NA>,22


## 4. EF ファイルの中身

EF ファイルは1行が 31 項目です。
1行には、ある日に算定した1つの項目が入ります。薬、検査、手術などの診療行為、毎日の入院基本料が、それぞれ1行になります。
同じ患者について、31 項目のうち8項目を日付順に出します。

In [5]:
ef_spec = load_spec("dpc_ef")
lines = m.ef.read_bytes().decode("cp932").splitlines()
raw_ef = pd.DataFrame([ln.split("\t") for ln in lines], columns=ef_spec.field_names())
cols = ["exec_date", "data_kubun", "receipt_code", "item_name", "quantity",
        "detail_flags", "act_points", "ward_code"]
raw_ef.loc[raw_ef["data_id"] == FOLLOW, cols].sort_values("exec_date", kind="stable")

,exec_date,data_kubun,receipt_code,item_name,quantity,detail_flags,act_points,ward_code
887,20260605,90,190999901,架空入院基本料,0000000.000,000000000000,100,W1
888,20260605,70,170999901,架空画像診断,0000000.000,000000000000,100,W1
902,20260605,21,620999901,架空錠Ａ ５ｍｇ,0000001.000,002200000000,0,W1
889,20260606,90,190999901,架空入院基本料,0000000.000,000000000000,100,W1
890,20260606,21,620999902,架空カプセルＢ,0000002.000,000000000000,100,W1
891,20260606,50,150999901,架空手術,0000000.000,000000000000,100,W1
892,20260607,90,190999901,架空入院基本料,0000000.000,000000000000,100,W1
893,20260608,90,190999901,架空入院基本料,0000000.000,000000000000,100,W2
894,20260608,21,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000,100,W2
895,20260608,21,620999902,架空カプセルＢ,0000002.000,000000000000,100,W2


各行は次のように読みます。

- `exec_date` は実施年月日です。入院は 6 月 5 日から 10 日までなので、6 日分の行があります。
- `data_kubun` はデータ区分で、項目の種類を表します。21 は内服、33 は注射、50 は手術、60 は検査、70 は画像診断、90 は入院基本料です。
- `receipt_code` はレセプト電算処理システム用コードです。実際のファイルでは基本マスターのコードが入ります。ここでは架空のコードを入れています。医薬品マスターのコードは 6 で、医科診療行為マスターのコードは 1 で始まります。架空のコードもこれに合わせています。
- `ward_code` はその日の病棟コードです。この患者は 6 月 8 日に W1 から W2 へ移っています。病棟コードは病院ごとに決めるコードです。
- `detail_flags` は行為明細区分情報で、12 桁あります。すべて 0 でない行が2つあります。
  - 6 月 5 日の `002200000000`。3 桁目の 2 は持参薬区分で、患者が持参した薬を、入院の契機となった傷病以外の治療に使ったことを表します。4 桁目の 2 は持参薬処方区分で、他院が処方した薬であることを表します。実施説明資料に従い、この行の行為点数（`act_points`）は 0 です。
  - 6 月 10 日の `100000000000`。1 桁目の 1 は退院時処方区分で、退院時に処方した薬であることを表します。

ファイル全体の行数を、項目の種類ごとに数えます。

In [6]:
from jpomop.spec import load_codes

kubun = load_codes("ef_data_kubun")
t = raw_ef.groupby(["data_kubun", "receipt_code", "item_name"]).size().rename("ef_lines").reset_index()
t.insert(1, "data_kubun_name", t["data_kubun"].map(kubun))
t

,data_kubun,data_kubun_name,receipt_code,item_name,ef_lines
0,21,内服,620999901,架空錠Ａ ５ｍｇ,258
1,21,内服,620999902,架空カプセルＢ,293
2,33,その他（注射）,640999903,架空注射液Ｃ ５００ｍＬ,245
3,50,手術,150999901,架空手術,43
4,60,検査・病理,160999901,架空検査（血液）,145
5,70,画像診断,170999901,架空画像診断,149
6,90,入院基本料,190999901,架空入院基本料,707


## 5. 読み込んで仕様と照合する

`read_yoshiki1` と `read_ef` は、ファイルを読み込む際に、すべての項目を仕様と照合します。
`pivot_yoshiki1` は、複数行に分かれた様式1 を、様式1 ごとに1行の表、病名ごとに1行の表、手術ごとに1行の表に組み直します。

架空データは仕様どおりに作ってあるので、違反は 0 件です。

In [7]:
from jpomop.parse.ef import read_ef
from jpomop.parse.yoshiki1 import pivot_yoshiki1, read_yoshiki1

y1 = read_yoshiki1(m.yoshiki1)
tables = pivot_yoshiki1(y1.table)
ef = read_ef(m.ef)

print("Form 1 lines:", y1.table.num_rows)
print("forms:", tables.forms.num_rows, " diagnoses:", tables.diagnoses.num_rows,
      " surgeries:", tables.surgeries.num_rows)
print("EF lines:", ef.table.num_rows)
print("violations:", len(y1.violations) + len(tables.violations) + len(ef.violations))

Form 1 lines: 900
forms: 120  diagnoses: 497  surgeries: 43
EF lines: 1840
violations: 0


組み直した後の、この患者の様式1 です。1枚だけあります。
郵便番号は残していません。

In [8]:
forms = tables.forms.to_pandas()
forms[forms["data_id"] == FOLLOW].T

,56
facility_code,999999999
data_id,0000000048
admission_date,2026-06-05
stay_no,0
summary_no,0
birth_date,1942-09-06
sex,1
admission_route,1
admission_type,100
ambulance,0


性別・入院経路・退院先・退院時転帰のコードは、実施説明資料に定義があります。
jpomop はこれらのコードを小さな表として持っています。
この患者は男性（1）で、家庭から入院し（1）、家庭へ退院して当院に通院します（1）。転帰は「不変」（4）です。

In [9]:
for name in ["y1_sex", "y1_admission_route", "y1_discharge_to", "y1_discharge_outcome"]:
    print(name, load_codes(name))

y1_sex {'1': '男', '2': '女'}
y1_admission_route {'0': '院内の他病棟からの転棟', '1': '家庭からの入院', '4': '他の病院・診療所の病棟からの転院', '5': '介護施設・福祉施設に入所中', '8': '院内で出生', '9': 'その他'}
y1_discharge_to {'0': '院内の他病棟への転棟', '1': '家庭への退院（当院に通院）', '2': '家庭への退院（他の病院・診療所に通院）', '3': '家庭への退院（その他）', '4': '他の病院・診療所への転院', '5': '介護老人保健施設に入所', '6': '介護老人福祉施設に入所', '7': '社会福祉施設、有料老人ホーム等に入所', '8': '終了（死亡等）', '9': 'その他', 'a': '介護医療院', 'b': '未転棟、未退院'}
y1_discharge_outcome {'1': '治癒・軽快', '3': '寛解', '4': '不変', '5': '増悪', '6': '最も医療資源を投入した傷病による死亡', '7': '最も医療資源を投入した傷病以外による死亡', '9': 'その他'}


同じ様式1 に入っている病名です。
`role` は、その病名が様式1 のどの行から来たかを表します。
`suspected` は、修飾語コード 8002 が付いていれば True になります。

In [10]:
dx = tables.diagnoses.to_pandas()
dx.loc[dx["data_id"] == FOLLOW,
       ["role", "seq", "icd10", "disease_code", "modifiers", "suspected", "disease_name"]]

,role,seq,icd10,disease_code,modifiers,suspected,disease_name
223,dx_main,0,C189,9000006,NaN,False,結腸癌
224,dx_trigger,0,C189,9000006,NaN,False,結腸癌
225,dx_resource1,0,C189,9000006,NaN,False,結腸癌
226,dx_comorbid,1,E785,9000103,NaN,False,脂質異常症
227,dx_comorbid,2,N189,9000104,NaN,False,慢性腎臓病
228,dx_complication,1,J690,9000201,NaN,False,誤嚥性肺炎


様式1 には手術も入っています。
jpomop はこの行を読み込みますが、CDM には書き出しません。
診療行為の表は EF ファイルから作ります。EF ファイルにも、6 月 6 日に同じ手術の行があります（8 章）。

In [11]:
sx = tables.surgeries.to_pandas()
sx[sx["data_id"] == FOLLOW]

,facility_code,data_id,admission_date,stay_no,summary_no,seq,surgery_date,surgery_code,surgery_name
22,999999999,0000000048,2026-06-05,0,0,1,2026-06-06,K9999,架空手術


## 6. 壊れたファイルを読ませる

照合の動きを見るため、次のセルで誤りを4つ入れたファイルを作ります。

1. 生年月日を 2 月 30 日にする
2. 性別のコードを、定義に無い 9 にする
3. 様式1 の1行を 16 項目にする
4. EF の実施年月日を、退院日より後にする

In [12]:
bad_lines = m.yoshiki1.read_bytes().decode("cp932").splitlines()

a = bad_lines[0].split("\t")   # first line: birth date and sex of patient 0000000001
a[8] = "19520230"               # 30 February
a[9] = "9"                      # sex code that is not in the manual
bad_lines[0] = "\t".join(a)

b = bad_lines[1].split("\t")   # second line: drop the last field
bad_lines[1] = "\t".join(b[:-1])

bad_y1 = work / "yoshiki1_bad.txt"
bad_y1.write_bytes(("\r\n".join(bad_lines) + "\r\n").encode("cp932"))

bad_ef_lines = m.ef.read_bytes().decode("cp932").splitlines()
c = bad_ef_lines[0].split("\t")
c[23] = "20260701"              # service date after discharge
bad_ef_lines[0] = "\t".join(c)
bad_ef = work / "ef_bad.txt"
_ = bad_ef.write_bytes(("\r\n".join(bad_ef_lines) + "\r\n").encode("cp932"))

誤り1つにつき1行が出ます。
ファイル名、行番号、項目、値、違反した規則を確認できます。

In [13]:
from jpomop.parse.cross import check_cross_file

r1 = read_yoshiki1(bad_y1)
t1 = pivot_yoshiki1(r1.table)
r2 = read_ef(bad_ef)
found = r1.violations + t1.violations + r2.violations
found += check_cross_file(r2.table, t1.forms, ef_file=bad_ef.name, y1_file=bad_y1.name)

pd.DataFrame([vars(v) for v in found])

,file,line,field,value,rule,message
0,yoshiki1_bad.txt,2,*,16,n_fields,expected 17 fields
1,yoshiki1_bad.txt,1,A000010.1:birth_date,19520230,date_invalid,not a calendar date
2,yoshiki1_bad.txt,1,A000010.2:sex,9,code_not_in_table,not in the code table
3,ef_bad.txt,1,exec_date,2026-07-01,exec_outside_stay,date of service is outside the stay


コマンドでも同じ照合ができます。

```sh
jpomop validate --yoshiki1 yoshiki1.txt --ef ef.txt --out violations.csv
```

違反が1件でもあると、コマンドは終了コード 1 で終わります。
照合するのは、jpomop が読み込む項目だけです。
提出用の形式チェックの代わりにはなりません。

## 7. OMOP CDM に変換する

`run_etl` は2つのファイルを読み込み、CDM の表を1つの DuckDB ファイルに書き出します。
コマンドでは `jpomop convert` が同じことをします。

処理結果の報告には、表ごとの行数と、変換しなかったものを理由ごとに数えた件数が入っています。

In [14]:
from jpomop.etl import run_etl

cdm_path = work / "cdm.duckdb"
report = run_etl([m.yoshiki1], [m.ef], cdm_path)

print(pd.Series(report.rows, name="rows").to_string())
print()
print("forms not used:", report.skipped_forms)
print("rows left out:", report.skipped_rows)
print("violations:", report.n_violations, " consistency problems:", report.invariant_problems)

person                  100
observation_period      100
visit_occurrence        120
visit_detail            143
death                     3
condition_occurrence    245
drug_exposure           777
procedure_occurrence    337

forms not used: {}
rows left out: {'condition_occurrence': {'duplicate_in_stay': 216, 'suspected': 36}, 'drug_exposure': {'brought_from_home': 19}}
violations: 0  consistency problems: []


報告の各項目を見ます。

- 患者は 100 人、入院は 120 回です。`person` が 100 行、`visit_occurrence` が 120 行になります。
- `duplicate_in_stay` は 216 件です。先ほど見た患者では、様式1 の同じ病名が3行に入っています。jpomop は、1回の入院の中で同じ ICD-10 を1行にまとめます。
- `suspected` は 36 件です。修飾語コード 8002 が付いた病名の行で、`condition_occurrence` には入れていません。
- `brought_from_home` は 19 件です。持参薬の行で、`drug_exposure` には入れていません。
- 「consistency problems」は、基本的な決まりを破った行の一覧です。たとえば、観察期間の外にある記録や、死亡日より後に始まる入院がこれに当たります。ここでは空です。

以下では、この患者について表を1つずつ見ます。

## 8. 1人の患者を表ごとに追いかける

### person

患者1人につき1行です。
`person_source_value` には、施設コードとデータ識別番号から計算したハッシュ値を入れています。元の番号は出力に残りません。
`gender_concept_id` の 8507 は、OMOP で男性を表す番号です。
DPC には人種や民族の項目が無いので、その番号は 0 にしています。

In [15]:
con = duckdb.connect(str(cdm_path), read_only=True)

con.sql(f"""
    SELECT person_id, person_source_value, gender_source_value, gender_concept_id,
           year_of_birth, month_of_birth, day_of_birth
    FROM person WHERE person_id = {FOLLOW_PERSON}
""").df()

,person_id,person_source_value,gender_source_value,gender_concept_id,year_of_birth,month_of_birth,day_of_birth
0,48,5cc23df4ab33c760,1,8507,1942,9,6


### visit_occurrence

入院1回につき1行です。
`visit_concept_id` の 9201 は入院（Inpatient Visit）を表します。
`visit_type_concept_id` の 32810 は、請求データ（Claim）から作った記録であることを表します。
2つの source value には DPC のコードをそのまま残しています。入院経路の 1 は「家庭からの入院」、退院先の 1 は「家庭への退院（当院に通院）」です。

In [16]:
con.sql(f"""
    SELECT visit_occurrence_id, visit_concept_id, visit_start_date, visit_end_date,
           visit_type_concept_id, admitted_from_source_value, discharged_to_source_value
    FROM visit_occurrence WHERE person_id = {FOLLOW_PERSON}
""").df()

,visit_occurrence_id,visit_concept_id,visit_start_date,visit_end_date,visit_type_concept_id,admitted_from_source_value,discharged_to_source_value
0,57,9201,2026-06-05,2026-06-10,32810,1,1


### visit_detail

同じ病棟にいた期間ごとに1行です。
この患者は、6 月 5 日から 7 日まで W1 に、8 日から 10 日まで W2 にいました。
2 行目の `preceding_visit_detail_id` は、1 行目を指しています。

In [17]:
con.sql(f"""
    SELECT visit_detail_id, visit_occurrence_id, visit_detail_start_date,
           visit_detail_end_date, visit_detail_source_value, preceding_visit_detail_id
    FROM visit_detail WHERE person_id = {FOLLOW_PERSON} ORDER BY visit_detail_start_date
""").df()

,visit_detail_id,visit_occurrence_id,visit_detail_start_date,visit_detail_end_date,visit_detail_source_value,preceding_visit_detail_id
0,71,57,2026-06-05,2026-06-07,W1,<NA>
1,72,57,2026-06-08,2026-06-10,W2,71


病棟の情報は、様式1 ではなく EF ファイルから取っています。
EF ファイルで、この入院の日ごとの病棟コードを並べると次のようになります。
上の2行と一致します。

In [18]:
(raw_ef[raw_ef["data_id"] == FOLLOW]
 .groupby("exec_date")["ward_code"].agg(lambda s: ",".join(sorted(set(s))))
 .to_frame().T)

exec_date,20260605,20260606,20260607,20260608,20260609,20260610
ward_code,W1,W1,W1,W2,W2,W2


### condition_occurrence

この患者の様式1 には、病名の行が6つありました。
C189 の3行は1行にまとまるので、この表では4行です。

In [19]:
con.sql(f"""
    SELECT condition_occurrence_id, condition_start_date, condition_source_value,
           condition_concept_id, condition_status_source_value, condition_status_concept_id
    FROM condition_occurrence WHERE person_id = {FOLLOW_PERSON}
""").df()

,condition_occurrence_id,condition_start_date,condition_source_value,condition_concept_id,condition_status_source_value,condition_status_concept_id
0,107,2026-06-05,C189,0,dx_main,32902
1,108,2026-06-05,E785,0,dx_comorbid,32907
2,109,2026-06-05,J690,0,dx_complication,32909
3,110,2026-06-05,N189,0,dx_comorbid,32907


様式1 には、病名ごとの日付がありません。
そのため `condition_start_date` には、どの行にも入院年月日を入れています。
入院後に発症した J690 では、この日付は実際の発症日より早くなります。

`condition_status_concept_id` は、その病名が様式1 のどの行から来たかを表します。

| 様式1 の行 | `condition_status_source_value` | `condition_status_concept_id` |
|---|---|---|
| 主傷病 | dx_main | 32902 Primary diagnosis |
| 入院時併存症 | dx_comorbid | 32907 Secondary admission diagnosis |
| 入院後発症疾患 | dx_complication | 32909 Secondary discharge diagnosis |

どの番号を当てるかは私の解釈で、公式に決まった対応ではありません。

### 疑い病名

この患者には疑い病名がありません。
次のセルで、疑い病名のある最初の患者を探し、その病名がどうなるかを見ます。

In [20]:
SUSPECTED = sorted(dx.loc[dx["suspected"], "data_id"].unique())
print(len(SUSPECTED), "patients have a suspected diagnosis. The first is", SUSPECTED[0])
print(dx.loc[dx["data_id"] == SUSPECTED[0],
             ["admission_date", "role", "icd10", "modifiers", "suspected", "disease_name"]]
      .to_string(index=False))
con.sql(f"""
    SELECT visit_occurrence_id, condition_start_date, condition_source_value,
           condition_status_source_value
    FROM condition_occurrence WHERE person_id = {int(SUSPECTED[0])}
""").df()

12 patients have a suspected diagnosis. The first is 0000000009
admission_date         role icd10 modifiers  suspected disease_name
    2026-06-06      dx_main  I500       NaN      False      うっ血性心不全
    2026-06-06   dx_trigger  I500       NaN      False      うっ血性心不全
    2026-06-06 dx_resource1  I500       NaN      False      うっ血性心不全
    2026-06-06  dx_comorbid   I10       NaN      False      本態性高血圧症
    2026-06-13      dx_main  J189      8002       True        肺炎の疑い
    2026-06-13   dx_trigger  J189      8002       True        肺炎の疑い
    2026-06-13 dx_resource1  J189      8002       True        肺炎の疑い


,visit_occurrence_id,condition_start_date,condition_source_value,condition_status_source_value
0,11,2026-06-06,I10,dx_comorbid
1,11,2026-06-06,I500,dx_main


データ識別番号 0000000009 の患者は2回入院しています。
2回目の入院の主傷病は J189 で、修飾語 8002 が付いているので「肺炎の疑い」です。
CDM の表には、1回目の入院の行しかありません。
2回目の入院には、病名の行が1つもありません。
確定していない病気を確定した病気として数えないための、意図した動きです。
その代わり、`condition_occurrence` から数え始める集計では、この入院が抜けます（12 章）。

### drug_exposure

この患者の、EF ファイルの薬の行です。

In [21]:
is_drug = raw_ef["receipt_code"].str.startswith("6")
raw_ef.loc[(raw_ef["data_id"] == FOLLOW) & is_drug,
           ["exec_date", "receipt_code", "item_name", "quantity", "detail_flags"]].sort_values("exec_date", kind="stable")

,exec_date,receipt_code,item_name,quantity,detail_flags
902,20260605,620999901,架空錠Ａ ５ｍｇ,0000001.000,002200000000
890,20260606,620999902,架空カプセルＢ,0000002.000,000000000000
894,20260608,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000
895,20260608,620999902,架空カプセルＢ,0000002.000,000000000000
897,20260609,640999903,架空注射液Ｃ ５００ｍＬ,0000001.000,000000000000
898,20260609,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000
901,20260610,640999903,架空注射液Ｃ ５００ｍＬ,0000001.000,000000000000
903,20260610,620999902,架空カプセルＢ,0000002.000,100000000000


薬の行は8行ありますが、下の表は7行です。
6 月 5 日の行は持参薬なので入れていません。
6 月 10 日の行は退院時処方なので、`stop_reason` に「discharge_prescription」を入れています。

In [22]:
con.sql(f"""
    SELECT drug_exposure_id, drug_exposure_start_date, drug_source_value, drug_concept_id,
           quantity, stop_reason, route_source_value, dose_unit_source_value
    FROM drug_exposure WHERE person_id = {FOLLOW_PERSON}
    ORDER BY drug_exposure_start_date, drug_exposure_id
""").df()

,drug_exposure_id,drug_exposure_start_date,drug_source_value,drug_concept_id,quantity,stop_reason,route_source_value,dose_unit_source_value
0,378,2026-06-06,620999902,0,2.0,NaN,21,009
1,379,2026-06-08,620999901,0,1.0,NaN,21,009
2,380,2026-06-08,620999902,0,2.0,NaN,21,009
3,381,2026-06-09,640999903,0,1.0,NaN,33,033
4,382,2026-06-09,620999901,0,1.0,NaN,21,009
5,383,2026-06-10,640999903,0,1.0,NaN,33,033
6,384,2026-06-10,620999902,0,2.0,discharge_prescription,21,009


開始日は EF ファイルの実施年月日です。
`quantity` は EF ファイルの使用量です。
`route_source_value` にはデータ区分（21 内服、33 注射）を、`dose_unit_source_value` には単位コードを残しています。

語彙を渡していないので、`drug_concept_id` は 0 です。
語彙を渡した場合の変換は 11 章で見ます。

### procedure_occurrence

EF ファイルのうち、コードが 1 で始まる行1つにつき1行です。入院基本料と食事は除いています。
この患者では、画像診断が2行（6 月 5 日と 9 日）と、手術が1行（6 月 6 日）です。
入院基本料の 6 行は入っていません。

`procedure_concept_id` は、今の版ではすべて 0 です。
日本の診療行為コードを標準の番号に対応づける処理は、まだありません。

In [23]:
con.sql(f"""
    SELECT procedure_occurrence_id, procedure_date, procedure_source_value,
           procedure_concept_id, visit_occurrence_id
    FROM procedure_occurrence WHERE person_id = {FOLLOW_PERSON} ORDER BY procedure_date
""").df()

,procedure_occurrence_id,procedure_date,procedure_source_value,procedure_concept_id,visit_occurrence_id
0,167,2026-06-05,170999901,0,57
1,168,2026-06-06,150999901,0,57
2,169,2026-06-09,170999901,0,57


### observation_period

患者1人につき1行です。ファイルに入っている最初の入院日から、最後の退院日までを期間にしています。
この患者は入院が1回なので、次のセルでは入院が2回ある最初の患者を出します。
期間は2回の入院をまたいでいます。

DPC のファイルから分かるのは、入院していた間のことだけです。
入院と入院の間は、実際には何も記録されていません。
観察期間の長さを使う研究では、この点に注意が要ります。

In [24]:
two_stays = con.sql("""
    SELECT min(person_id) FROM (SELECT person_id FROM visit_occurrence
                                GROUP BY 1 HAVING count(*) > 1)
""").fetchone()[0]
con.sql(f"""
    SELECT o.person_id, o.observation_period_start_date, o.observation_period_end_date,
           v.visit_occurrence_id, v.visit_start_date, v.visit_end_date
    FROM observation_period o JOIN visit_occurrence v USING (person_id)
    WHERE o.person_id = {two_stays} ORDER BY v.visit_start_date
""").df()

,person_id,observation_period_start_date,observation_period_end_date,visit_occurrence_id,visit_start_date,visit_end_date
0,6,2026-06-09,2026-06-25,6,2026-06-09,2026-06-14
1,6,2026-06-09,2026-06-25,7,2026-06-19,2026-06-25


## 9. 死亡

DPC のデータに死亡が残るのは、入院中に亡くなった場合だけです。
jpomop は、様式1 の退院時転帰が 6 か 7 のときに `death` の行を作ります。
死亡日には退院年月日を入れています。

In [25]:
con.sql("""
    SELECT d.person_id, d.death_date, d.death_type_concept_id,
           v.visit_start_date, v.visit_end_date, v.discharged_to_source_value
    FROM death d JOIN visit_occurrence v
      ON v.person_id = d.person_id AND v.visit_end_date = d.death_date
    ORDER BY d.person_id
""").df()

,person_id,death_date,death_type_concept_id,visit_start_date,visit_end_date,discharged_to_source_value
0,25,2026-06-14,32810,2026-06-11,2026-06-14,8
1,56,2026-06-15,32810,2026-06-11,2026-06-15,8
2,70,2026-06-26,32810,2026-06-18,2026-06-26,8


3 件の死亡の元になった様式1 です。
3 件とも、退院先が 8（終了・死亡等）で、転帰が 7 です。
転帰 6 は、医療資源を最も投入した傷病による死亡です。
転帰 7 は、それ以外の原因による死亡です。
退院した後の死亡は DPC のデータに無いので、`death` の表にも入りません。

In [26]:
forms.loc[forms["discharge_outcome"].isin(["6", "7"]),
          ["data_id", "admission_date", "discharge_date", "discharge_to", "discharge_outcome"]]

,data_id,admission_date,discharge_date,discharge_to,discharge_outcome
30,0000000025,2026-06-11,2026-06-14,8,7
66,0000000056,2026-06-11,2026-06-15,8,7
84,0000000070,2026-06-18,2026-06-26,8,7


## 10. 数えるだけで変換しない様式1

病院は、1回の入院について様式1 を複数作ることがあります。
たとえば、患者が病棟グループをまたいで移ると、入院全体の様式1 に加えて、病棟グループごとの様式1 を作ります。
入院全体の様式1 は `summary_no`（統括診療情報番号）が 0 です。
病棟グループごとの様式1 は 1、2 と続きます。

架空データには、入院全体の様式1 しかありません。
そこで次のセルでは、追いかけている患者の様式1 を写し、`summary_no` を 1 に変えてファイルに足します。
病棟グループごとの様式1 を、大まかにまねたものです。

In [27]:
src = m.yoshiki1.read_bytes().decode("cp932").splitlines()
extra = []
for ln in src:
    f = ln.split("\t")
    if f[1] == FOLLOW:
        f[4] = "1"              # 統括診療情報番号: 0 -> 1 (ward-group form)
        extra.append("\t".join(f))

y1_more = work / "yoshiki1_with_ward_group_form.txt"
y1_more.write_bytes(("\r\n".join(src + extra) + "\r\n").encode("cp932"))

report2 = run_etl([y1_more], [m.ef], work / "cdm2.duckdb")
print("forms not used:", report2.skipped_forms)
print("visit_occurrence rows:", report2.rows["visit_occurrence"],
      " condition_occurrence rows:", report2.rows["condition_occurrence"])

forms not used: {'summary_no=1': 1}
visit_occurrence rows: 120  condition_occurrence rows: 245


足した様式1 は「forms not used」に数えられています。
入院の数と病名の行数は前と同じ（120 と 245）なので、入院を二重に数えていません。

7 日以内の再入院をまとめた様式1（A と B）も、同じ扱いにしています。
入院 90 日目の仮の様式1（Z）は、その入院にまだ入院全体の様式1 が無いときだけ使います。
この扱いが正しいかどうかは、実データではまだ確かめていません。

## 11. concept ID と語彙

ここまでの変換では、病名・薬・診療行為の concept ID はすべて 0 です。
元のコードは source value の列に残っています。

In [28]:
con.sql("""
    SELECT 'condition_occurrence' AS cdm_table, count(*) AS n_rows,
           count(*) FILTER (WHERE condition_concept_id = 0) AS concept_id_0
    FROM condition_occurrence
    UNION ALL SELECT 'drug_exposure', count(*),
           count(*) FILTER (WHERE drug_concept_id = 0) FROM drug_exposure
    UNION ALL SELECT 'procedure_occurrence', count(*),
           count(*) FILTER (WHERE procedure_concept_id = 0) FROM procedure_occurrence
""").df()

,cdm_table,n_rows,concept_id_0
0,condition_occurrence,245,245
1,drug_exposure,777,777
2,procedure_occurrence,337,337


concept ID を入れるには、次の3つのファイルが要ります。どれも、使う人が自分でダウンロードします。

- Athena の語彙（ICD10、SNOMED、RxNorm、RxNorm Extension）
- 医薬品マスター（薬のレセプト電算処理システム用コードと、薬価基準収載医薬品コードの対応が入っています）
- 木村らの対応表（2025 年。薬価基準収載医薬品コードと OMOP の concept の対応が入っています）

Athena の語彙一式は大きく、ダウンロードには無料のアカウント登録が要るので、このノートブックには入れていません。
代わりに次のセルには、架空データに出てくる 12 種類の ICD-10 コードの分だけを抜き出して入れています。
この行は本物で、2026 年 10 月 14 日に OMOP の標準語彙から写したものです。
実際に使うときは、https://athena.ohdsi.org から語彙一式をダウンロードしてください。

ICD-10 のコード（標準でない concept）は、「Maps to」で SNOMED の標準の concept につながっています。
J69.0 は、標準の concept 2つにつながっています。

In [29]:
# Rows copied from the OMOP standard vocabularies on 14 October 2026.
# (concept_id, concept_code, vocabulary_id, concept_class_id, standard_concept, concept_name)
CONCEPTS = [
    (45561790, 'C18.9', 'ICD10', 'ICD10 code', '', 'Malignant neoplasm: Colon, unspecified'),
    (197500, '93761005', 'SNOMED', 'Disorder', 'S', 'Primary malignant neoplasm of colon'),
    (45561952, 'E11.9', 'ICD10', 'ICD10 code', '', 'Type 2 diabetes mellitus, without complications'),
    (201826, '44054006', 'SNOMED', 'Disorder', 'S', 'Type 2 diabetes mellitus'),
    (45537986, 'E78.5', 'ICD10', 'ICD10 code', '', 'Hyperlipidaemia, unspecified'),
    (432867, '55822004', 'SNOMED', 'Disorder', 'S', 'Hyperlipidemia'),
    (45591453, 'I10', 'ICD10', 'ICD10 Hierarchy', '', 'Essential (primary) hypertension'),
    (320128, '59621000', 'SNOMED', 'Disorder', 'S', 'Essential hypertension'),
    (45567179, 'I50.0', 'ICD10', 'ICD10 code', '', 'Congestive heart failure'),
    (319835, '42343007', 'SNOMED', 'Disorder', 'S', 'Congestive heart failure'),
    (45596216, 'I63.9', 'ICD10', 'ICD10 code', '', 'Cerebral infarction, unspecified'),
    (443454, '432504007', 'SNOMED', 'Disorder', 'S', 'Cerebral infarction'),
    (45601123, 'J18.9', 'ICD10', 'ICD10 code', '', 'Pneumonia, unspecified'),
    (255848, '233604007', 'SNOMED', 'Disorder', 'S', 'Pneumonia'),
    (45591562, 'J69.0', 'ICD10', 'ICD10 code', '', 'Pneumonitis due to food and vomit'),
    (4112839, '196033004', 'SNOMED', 'Disorder', 'S', 'Pneumonitis caused by inhalation of regurgitated food'),
    (4110180, '196035006', 'SNOMED', 'Disorder', 'S', 'Pneumonitis caused by inhalation of vomitus'),
    (45552951, 'K80.2', 'ICD10', 'ICD10 code', '', 'Calculus of gallbladder without cholecystitis'),
    (36716712, '722869007', 'SNOMED', 'Disorder', 'S', 'Calculus of gallbladder without cholecystitis or cholangitis'),
    (45553438, 'N18.9', 'ICD10', 'ICD10 code', '', 'Chronic kidney disease, unspecified'),
    (46271022, '709044004', 'SNOMED', 'Disorder', 'S', 'Chronic kidney disease'),
    (45548665, 'N39.0', 'ICD10', 'ICD10 code', '', 'Urinary tract infection, site not specified'),
    (81902, '68566005', 'SNOMED', 'Disorder', 'S', 'Urinary tract infectious disease'),
    (1529272, 'S72.00', 'ICD10', 'ICD10 code', '', 'Fracture of neck of femur, closed'),
    (434500, '359820003', 'SNOMED', 'Disorder', 'S', 'Closed fracture of neck of femur'),
]
# (ICD10 concept_id, standard concept_id) pairs with the relationship "Maps to"
MAPS_TO = [
    (45561790, 197500), (45561952, 201826), (45537986, 432867),
    (45591453, 320128), (45567179, 319835), (45596216, 443454),
    (45601123, 255848), (45591562, 4112839), (45591562, 4110180),
    (45552951, 36716712), (45553438, 46271022), (45548665, 81902),
    (1529272, 434500),
]

athena = work / "athena_extract"
athena.mkdir()
concept_lines = ["concept_id\tconcept_name\tdomain_id\tvocabulary_id\tconcept_class_id\t"
                 "standard_concept\tconcept_code\tvalid_start_date\tvalid_end_date\tinvalid_reason"]
for cid, code, vocabulary, cls, standard, name in CONCEPTS:
    concept_lines.append(f"{cid}\t{name}\tCondition\t{vocabulary}\t{cls}\t{standard}\t{code}"
                         "\t19700101\t20991231\t")
(athena / "CONCEPT.csv").write_text("\n".join(concept_lines) + "\n", encoding="utf-8")
relationship_lines = ["concept_id_1\tconcept_id_2\trelationship_id\tvalid_start_date\t"
                      "valid_end_date\tinvalid_reason"]
for source, target in MAPS_TO:
    relationship_lines.append(f"{source}\t{target}\tMaps to\t19700101\t20991231\t")
_ = (athena / "CONCEPT_RELATIONSHIP.csv").write_text("\n".join(relationship_lines) + "\n",
                                                     encoding="utf-8")

pd.DataFrame(CONCEPTS, columns=["concept_id", "concept_code", "vocabulary_id",
                                "concept_class_id", "standard_concept", "concept_name"])

,concept_id,concept_code,vocabulary_id,concept_class_id,standard_concept,concept_name
0,45561790,C18.9,ICD10,ICD10 code,,"Malignant neoplasm: Colon, unspecified"
1,197500,93761005,SNOMED,Disorder,S,Primary malignant neoplasm of colon
2,45561952,E11.9,ICD10,ICD10 code,,"Type 2 diabetes mellitus, without complications"
3,201826,44054006,SNOMED,Disorder,S,Type 2 diabetes mellitus
4,45537986,E78.5,ICD10,ICD10 code,,"Hyperlipidaemia, unspecified"
5,432867,55822004,SNOMED,Disorder,S,Hyperlipidemia
6,45591453,I10,ICD10,ICD10 Hierarchy,,Essential (primary) hypertension
7,320128,59621000,SNOMED,Disorder,S,Essential hypertension
8,45567179,I50.0,ICD10,ICD10 code,,Congestive heart failure
9,319835,42343007,SNOMED,Disorder,S,Congestive heart failure


薬は事情が違います。
架空データの薬のコードは作り物なので、本物の医薬品マスターにも木村らの対応表にも当たりません。
そこで次のセルでは、同じ形をした架空のファイルを2つ作ります。
中の concept ID は 9000 で始まる番号で、OMOP の語彙には存在しません。
対応が見つからないコードの動きも見るため、3 種類目の薬にはわざと対応を入れていません。

In [30]:
# A file shaped like the drug master: 42 fields, receipt code in field 3,
# NHI drug price code in field 32. Both codes are invented.
pairs = {"620999901": "9999001F1011", "620999902": "9999002M1011", "640999903": "9999400A1011"}
rows = []
for receipt, price_code in pairs.items():
    fields = ["0", "Y", receipt] + [""] * 39
    fields[31] = price_code
    rows.append(",".join(f'"{x}"' for x in fields))
drug_master = work / "y_invented.csv"
drug_master.write_bytes(("\r\n".join(rows) + "\r\n").encode("cp932"))

# A file shaped like rxnorm_mapping.tsv. The concept IDs are invented (9000...).
# The third drug has no match on purpose.
mapping = work / "rxnorm_mapping_invented.tsv"
_ = mapping.write_text("Category\tNHIDPLC \tOSV_ID\tOSV_NAME\n"
                       "PO\t9999001F1011\t900001001\tinvented drug A\n"
                       "PO\t9999002M1011\t900001002\tinvented drug B\n"
                       "Injection\t9999400A1011\tN/A\tN/A\n", encoding="utf-8")

`build_vocab` は、3つのファイルを1つの小さな DuckDB ファイルにまとめます。

DPC の ICD-10 にはピリオドが無く、5 桁目が付くこともあります。
jpomop は、細かいコードから順に対応を探します。
S7200 は S72.00（Fracture of neck of femur, closed）として見つかります。
J690 からは、標準の concept が2つ返ります。

In [31]:
from jpomop.vocab import Vocab, build_vocab, icd10_candidates

vocab_path = work / "vocab.duckdb"
vr = build_vocab(athena, [drug_master], mapping, vocab_path)
print("ICD-10 codes in the extract:", vr.icd10_codes, " with a standard concept:", vr.icd10_mapped)
print("drugs in the drug master:", vr.receipt_drugs, " with a concept:", vr.receipt_drugs_mapped)
print()

v = Vocab.load(vocab_path)
for code in ["C189", "S7200", "J690"]:
    hits = v.map_icd10_all(code)
    print(code, "tries", icd10_candidates(code), "-> ICD10 concept", hits[0].source_concept_id,
          "-> standard concept(s)", [h.concept_id for h in hits])
for code in ["620999901", "640999903"]:
    print(code, "-> drug concept", v.map_drug(code).concept_id)

ICD-10 codes in the extract: 12  with a standard concept: 12
drugs in the drug master: 3  with a concept: 2

C189 tries ['C18.9', 'C18'] -> ICD10 concept 45561790 -> standard concept(s) [197500]
S7200 tries ['S72.00', 'S72.0', 'S72'] -> ICD10 concept 1529272 -> standard concept(s) [434500]
J690 tries ['J69.0', 'J69'] -> ICD10 concept 45591562 -> standard concept(s) [4110180, 4112839]
620999901 -> drug concept 900001001
640999903 -> drug concept 0


語彙を渡して、同じ変換をもう一度します。
`condition_source_concept_id` が ICD-10 の concept、`condition_concept_id` がその対応先の標準の concept です。

J690 は、入院1回につき2行になりました。標準の concept 1つにつき1行です。
1つのコードが複数の concept に対応するときの、OMOP の決まりに合わせています。
そのため、ここでの `condition_occurrence` は 259 行で、語彙なしのとき（245 行）より 14 行多くなっています。

In [32]:
cdm3 = work / "cdm_with_vocab.duckdb"
report3 = run_etl([m.yoshiki1], [m.ef], cdm3, vocab=vocab_path)

con3 = duckdb.connect(str(cdm3), read_only=True)
con3.sql(f"""
    SELECT c.condition_source_value, c.condition_source_concept_id, c.condition_concept_id,
           n.concept_name AS standard_concept_name, count(*) AS n_rows
    FROM condition_occurrence c
    LEFT JOIN read_csv('{athena / "CONCEPT.csv"}', delim='\t', header=true, quote='',
                       all_varchar=true) n
      ON n.concept_id = CAST(c.condition_concept_id AS VARCHAR)
    GROUP BY ALL ORDER BY c.condition_source_value
""").df()

,condition_source_value,condition_source_concept_id,condition_concept_id,standard_concept_name,n_rows
0,C189,45561790,197500,Primary malignant neoplasm of colon,12
1,E119,45561952,201826,Type 2 diabetes mellitus,27
2,E785,45537986,432867,Hyperlipidemia,28
3,I10,45591453,320128,Essential hypertension,35
4,I500,45567179,319835,Congestive heart failure,15
5,I639,45596216,443454,Cerebral infarction,17
6,J189,45601123,255848,Pneumonia,12
7,J690,45591562,4112839,Pneumonitis caused by inhalation of regurgitated food,14
8,J690,45591562,4110180,Pneumonitis caused by inhalation of vomitus,14
9,K802,45552951,36716712,Calculus of gallbladder without cholecystitis or cholangitis,18


`coverage` は、0 でない concept ID が入った行数と、入ったコードの種類数を数えます。
入らなかったコードは、件数の多い順に並びます。
コマンドでは `jpomop report` で同じ表が出ます。

語彙一式を使えば、対応づけを急ぐべきコードをこの表で確認できます。
病名が 100% なのは、架空データの 12 種類に合わせて抜き出したからにすぎません。
薬の割合は、架空のファイルの結果です。
どちらも、実データでの割合を表すものではありません。
診療行為の対応づけはまだ無いので、0% のままです。

In [33]:
from jpomop.report import coverage

pd.DataFrame(
    [(c.table, c.rows, c.mapped_rows, f"{c.row_rate:.0%}", c.codes, c.mapped_codes, c.top_unmapped)
     for c in coverage(cdm3)],
    columns=["cdm_table", "rows", "mapped_rows", "row_rate", "codes", "mapped_codes", "unmapped_codes"],
)

,cdm_table,rows,mapped_rows,row_rate,codes,mapped_codes,unmapped_codes
0,condition_occurrence,259,259,100%,12,12,[]
1,drug_exposure,777,532,68%,3,2,"[(640999903, 245)]"
2,procedure_occurrence,337,0,0%,3,0,"[(170999901, 149), (160999901, 145), (150999901, 43)]"


## 12. 表に問い合わせる例を4つ

表は DuckDB ファイルに入っているので、SQL で集計できます。
データが架空なので、下の結果自体に意味はありません。
ここでは、表どうしが想定どおりにつながるかを確かめます。

### 主傷病ごとの入院日数

`condition_status_concept_id` が 32902 の行が主傷病です。

In [34]:
con.sql("""
    SELECT c.condition_source_value AS main_diagnosis_icd10,
           count(*) AS stays,
           round(avg(v.visit_end_date - v.visit_start_date + 1), 1) AS mean_days,
           min(v.visit_end_date - v.visit_start_date + 1) AS min_days,
           max(v.visit_end_date - v.visit_start_date + 1) AS max_days
    FROM visit_occurrence v
    JOIN condition_occurrence c USING (visit_occurrence_id)
    WHERE c.condition_status_concept_id = 32902
    GROUP BY 1 ORDER BY stays DESC
""").df()

,main_diagnosis_icd10,stays,mean_days,min_days,max_days
0,S7200,19,6.4,2,10
1,K802,18,5.1,2,10
2,I639,17,6.1,2,10
3,N390,15,5.8,2,10
4,I500,15,5.0,2,9
5,C189,12,6.1,2,10
6,J189,12,7.4,4,10


入院の数を足すと 108 で、120 になりません。
残りの 12 回は主傷病に疑いが付いているので、主傷病の行がありません。
疑い病名を外したことが、集計結果にも表れています。
すべての入院を数えたい場合は、`visit_occurrence` から集計を始める必要があります。

### 薬ごとの患者数と、退院時処方の行数

In [35]:
con.sql("""
    SELECT d.drug_source_value,
           count(DISTINCT d.person_id) AS persons,
           count(DISTINCT d.visit_occurrence_id) AS stays,
           count(*) AS rows_,
           count(*) FILTER (WHERE d.stop_reason = 'discharge_prescription') AS at_discharge
    FROM drug_exposure d GROUP BY 1 ORDER BY 1
""").df()

,drug_source_value,persons,stays,rows_,at_discharge
0,620999901,87,103,239,0
1,620999902,98,115,293,46
2,640999903,94,109,245,0


### 病棟を移った入院の数

In [36]:
con.sql("""
    SELECT wards_in_stay, count(*) AS stays
    FROM (SELECT visit_occurrence_id, count(*) AS wards_in_stay
          FROM visit_detail GROUP BY 1)
    GROUP BY 1 ORDER BY 1
""").df()

,wards_in_stay,stays
0,1,97
1,2,23


### 2回以上入院した患者と、前の退院からの日数

In [37]:
con.sql("""
    SELECT person_id, visit_occurrence_id, visit_start_date, visit_end_date,
           visit_start_date - lag(visit_end_date) OVER (PARTITION BY person_id
                                                         ORDER BY visit_start_date) AS days_since_last_discharge
    FROM visit_occurrence
    QUALIFY count(*) OVER (PARTITION BY person_id) > 1
    ORDER BY person_id, visit_start_date
    LIMIT 8
""").df()

,person_id,visit_occurrence_id,visit_start_date,visit_end_date,days_since_last_discharge
0,6,6,2026-06-09,2026-06-14,<NA>
1,6,7,2026-06-19,2026-06-25,5
2,7,8,2026-06-07,2026-06-14,<NA>
3,7,9,2026-06-16,2026-06-19,2
4,9,11,2026-06-06,2026-06-09,<NA>
5,9,12,2026-06-13,2026-06-15,4
6,14,17,2026-06-10,2026-06-16,<NA>
7,14,18,2026-06-19,2026-06-28,3


データ識別番号 7、9、14 の患者は、7 日以内に再入院しています。
実際のファイルでは、実施説明資料の条件に当てはまれば、病院は再入院をまとめた様式1（A）も作ります。
架空データでは、この様式1 は作っていません。

## 13. Parquet で書き出す

同じ表を、表ごとに1つの Parquet ファイルとして書き出すこともできます。

In [38]:
report4 = run_etl([m.yoshiki1], [m.ef], work / "cdm_parquet", fmt="parquet")
sorted(p.name for p in (work / "cdm_parquet").iterdir())

['condition_occurrence.parquet',
 'death.parquet',
 'drug_exposure.parquet',
 'observation_period.parquet',
 'person.parquet',
 'procedure_occurrence.parquet',
 'visit_detail.parquet',
 'visit_occurrence.parquet']

## 14. このノートブックで分からないこと

- 病院の実データで jpomop がどう動くかは分かりません。実データではまだ動かしていません。
- 実際の変換率は分かりません。11 章の病名の語彙は架空データの 12 種類に合わせた抜き出しで、薬のファイルは架空です。
- 診療行為には、まだ標準の concept が入りません。
- 点数や金額は変換していません。
- 外来のファイル、D ファイル、H ファイルは読み込みません。
- 病棟グループごとの様式1、再入院をまとめた様式1、仮の様式1 の扱いは、実施説明資料を私が読んで決めたものです。実データでの確認が要ります。

DPC データを扱っている方に、データのある環境で jpomop を動かしていただけると助かります。
データを外に出す必要はありません。件数とエラーの内容だけで十分です。
Issue は https://github.com/kojikojikojikoji/jpomop/issues で受け付けています。

下のセルで、ファイルを閉じて一時フォルダを消します。

In [39]:
con.close()
con3.close()
import shutil

shutil.rmtree(work)